# AdaptiveHb — Synthetic Smoke Experiment (no GPU / no PyTorch)

This notebook verifies the **entire framework end-to-end** without any heavy ML dependencies. It generates a small synthetic, spec-conformant dataset, then runs a full **baseline-vs-adaptive** experiment on the built-in reference models and shows the archived comparison (with paired significance), the reproducibility provenance manifest, and the generated figures.

It runs anywhere Python 3.11+ is available — use it to confirm your checkout is healthy before launching the real, PyTorch-backed run in `train_pipeline.ipynb`.

> With the torch-free reference models every tissue returns a constant prediction, so `baseline == adaptive` here (improvement 0.0). That is expected — the point of this notebook is to prove the *machinery* runs and archives correctly.

## Get the code (Kaggle / Colab / local)

Run the cell below **first**. On Kaggle or Colab it clones the repository and enters it; when you already run from inside a local clone it does nothing.

In [ ]:
# === Get the code — run this FIRST (Kaggle / Colab / local) ==================
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/junaidmaqbool/AgenticHb.git"


def _has_repo(path) -> bool:
    return (Path(path) / "configs" / "project.yaml").is_file()


if _has_repo(Path.cwd()):
    pass                                   # already inside the repo (local run)
elif _has_repo(Path.cwd().parent):
    os.chdir(Path.cwd().parent)            # notebook opened from notebooks/
elif Path("AgenticHb").exists() and _has_repo("AgenticHb"):
    os.chdir("AgenticHb")                  # cloned earlier in this session
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "AgenticHb"], check=True)
    os.chdir("AgenticHb")

print("Working directory:", Path.cwd())
assert _has_repo(Path.cwd()), "Repository not found - check the clone step above."


## 0. Setup

In [ ]:
# --- Make the framework importable and locate the repository root -------------
# Works whether you `pip install -e .` the package or just run from a clone.
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward from `start` (default: cwd) until a folder with configs/project.yaml."""
    start = (start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        if (directory / "configs" / "project.yaml").is_file():
            return directory
    raise FileNotFoundError(
        "Could not locate the repository root (a folder with configs/project.yaml). "
        "Run this notebook from inside the AgenticHb repository."
    )


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))  # allows running without `pip install`

CONFIG_DIR = REPO_ROOT / "configs"
print("Repository root :", REPO_ROOT)
print("Config directory:", CONFIG_DIR)


In [ ]:
import json
import tempfile
from pathlib import Path

from adaptivehb.pipeline import HbPipeline
from adaptivehb.dataset import generate_synthetic_dataset

# All inputs are variables — nothing is hardcoded inside the framework.
NUM_PATIENTS = 16   # more patients => a larger held-out test split
EPOCHS = 2          # reference models train instantly; keep this small
SEED = 7
print('adaptivehb imported OK')

## 1. Generate a synthetic dataset

In [ ]:
WORKDIR = Path(tempfile.mkdtemp(prefix='adaptivehb_smoke_'))
DATASET_ROOT = WORKDIR / 'dataset'
generate_synthetic_dataset(DATASET_ROOT, num_patients=NUM_PATIENTS, seed=SEED)
print('Working directory:', WORKDIR)
print('Dataset root     :', DATASET_ROOT)
print('Contents         :', sorted(p.name for p in DATASET_ROOT.iterdir()))

## 2. Run the full experiment

`HbPipeline.experiment(...)` trains every model, evaluates a **static baseline** against the **adaptive** (agent-fused) pipeline on the held-out test split, and archives metrics, the comparison, per-sample predictions, figures, a provenance manifest, and a summary into a fresh experiment directory.

In [ ]:
pipeline = HbPipeline.from_config_dir(CONFIG_DIR, base_dir=WORKDIR, dataset_root=DATASET_ROOT)
result = pipeline.experiment('synthetic_smoke', epochs=EPOCHS)
print('Experiment id :', result.experiment_id)
print('Archived at   :', result.root)

## 3. Baseline vs adaptive comparison (with paired significance)

In [ ]:
comparison = result.comparison
print('metric         :', comparison['metric'])
print('baseline MAE   :', round(comparison['baseline'], 4))
print('adaptive MAE   :', round(comparison['adaptive'], 4))
print('improvement    :', round(comparison['improvement'], 4))
print('adaptive_better:', comparison['adaptive_better'])

sig = comparison.get('significance')
if sig:
    print('\n-- paired significance --')
    print('n_pairs         :', sig['n_pairs'])
    print('paired t-test p :', round(sig['paired_t_test']['p_value'], 4))
    print('wilcoxon p      :', round(sig['wilcoxon']['p_value'], 4))
    print("cohen's d       :", round(sig['cohens_d'], 4))
    ci = sig['bootstrap_ci']
    print('bootstrap 95% CI:', (round(ci['ci_lower'], 4), round(ci['ci_upper'], 4)))

## 4. Reproducibility provenance manifest

In [ ]:
prov = result.provenance
print('framework version:', prov['framework_version'])
print('seed             :', prov['seed'])
print('python           :', prov['environment']['python_version'])
print('packages         :', prov['environment']['packages'])
print('git              :', prov['git'])  # None outside a git checkout
print('config digest    :', prov['config']['digest'])
print('dataset          :', {k: prov['dataset'][k] for k in ('num_patients', 'num_samples', 'split_sizes')})

## 5. Generated figures

In [ ]:
from IPython.display import Image, display

figure_dir = Path(result.root) / 'figures'
pngs = sorted(figure_dir.glob('*.png')) if figure_dir.is_dir() else []
print('figures:', [p.name for p in pngs])
for png in pngs:
    display(Image(filename=str(png)))

## Done

If this notebook ran to the end, the framework is healthy on your machine: training, registry/checkpointing, the agent workflow, evaluation, significance testing, provenance, and reporting all executed and archived. Next, open **`train_pipeline.ipynb`** to run the real, PyTorch-backed experiment on your dataset.